Задача: реализовать и запустить модель LDA и Gibbs Sampling с числов тегов 20. Вывести топ-10 слов по каждому тегу. Соотнести полученные теги с тегами из датасета.

In [7]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer

print("Загрузка данных...")
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
documents_subset = newsgroups_train.data[:2000]

print("Векторизация...")
vectorizer = CountVectorizer(
    max_features=3000,
    stop_words='english',
    min_df=8,
    max_df=0.6
)
X = vectorizer.fit_transform(documents_subset)
vocab = vectorizer.get_feature_names_out()
V = len(vocab)
print(f"Словарь: {V} слов")

print("Подготовка данных...")
M = X.shape[0] 

all_words = []
all_doc_ids = []
for doc_id in range(M):
    doc = X[doc_id]
    for word_idx, count in zip(doc.indices, doc.data):
        all_words.extend([word_idx] * int(count))
        all_doc_ids.extend([doc_id] * int(count))

all_words = np.array(all_words)
all_doc_ids = np.array(all_doc_ids)
W = len(all_words)
print(f"Всего слов: {W}")


K = 20
alpha = 0.1
beta = 0.01
n_iter = 100
beta_sum = beta * V


n_dk = np.zeros((M, K), dtype=np.int32)
n_kw = np.zeros((K, V), dtype=np.int32)
n_k = np.zeros(K, dtype=np.int32)      

z = np.random.randint(0, K, size=W)

for i in range(W):
    doc_id = all_doc_ids[i]
    word_id = all_words[i]
    topic = z[i]
    n_dk[doc_id, topic] += 1
    n_kw[topic, word_id] += 1
    n_k[topic] += 1

print("Начало Gibbs Sampling...")

alpha_vec = np.full(K, alpha)

for iteration in range(n_iter):
    print(f"Итерация {iteration+1}/{n_iter}")
    
    order = np.random.permutation(W)
    
    for i in order:
        doc_id = all_doc_ids[i]
        word_id = all_words[i]
        old_topic = z[i]
        
        n_dk[doc_id, old_topic] -= 1
        n_kw[old_topic, word_id] -= 1
        n_k[old_topic] -= 1   
        
        p = (n_dk[doc_id, :] + alpha_vec) * (n_kw[:, word_id] + beta) / (n_k + beta_sum)
        p_sum = np.sum(p)
        if p_sum > 0:
            p /= p_sum
        else:
            p = np.ones(K) / K
            
        new_topic = np.random.choice(K, p=p)
        z[i] = new_topic
        
        n_dk[doc_id, new_topic] += 1
        n_kw[new_topic, word_id] += 1
        n_k[new_topic] += 1

phi = (n_kw + beta) / (n_k[:, np.newaxis] + beta_sum)

print("\nТоп-10 слов по темам:")
for k in range(K):
    top_indices = np.argpartition(phi[k], -10)[-10:]
    top_indices = top_indices[np.argsort(phi[k][top_indices])[::-1]]
    top_words = [vocab[i] for i in top_indices]
    print(f"Тема {k:2d}: {', '.join(top_words)}")

Загрузка данных...
Векторизация...
Словарь: 3000 слов
Подготовка данных...
Всего слов: 117985
Начало Gibbs Sampling...
Итерация 1/100
Итерация 2/100
Итерация 3/100
Итерация 4/100
Итерация 5/100
Итерация 6/100
Итерация 7/100
Итерация 8/100
Итерация 9/100
Итерация 10/100
Итерация 11/100
Итерация 12/100
Итерация 13/100
Итерация 14/100
Итерация 15/100
Итерация 16/100
Итерация 17/100
Итерация 18/100
Итерация 19/100
Итерация 20/100
Итерация 21/100
Итерация 22/100
Итерация 23/100
Итерация 24/100
Итерация 25/100
Итерация 26/100
Итерация 27/100
Итерация 28/100
Итерация 29/100
Итерация 30/100
Итерация 31/100
Итерация 32/100
Итерация 33/100
Итерация 34/100
Итерация 35/100
Итерация 36/100
Итерация 37/100
Итерация 38/100
Итерация 39/100
Итерация 40/100
Итерация 41/100
Итерация 42/100
Итерация 43/100
Итерация 44/100
Итерация 45/100
Итерация 46/100
Итерация 47/100
Итерация 48/100
Итерация 49/100
Итерация 50/100
Итерация 51/100
Итерация 52/100
Итерация 53/100
Итерация 54/100
Итерация 55/100
Итерация 5